In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
bronze_orders = spark.table("bronze.orders")

In [0]:
orders_df = bronze_orders.select(
    col("order_id").cast("string").alias("order_id"),
    col("warehouse_id").cast("string").alias("warehouse_id"),
    col("created_ts").cast("timestamp").alias("created_ts"),
    col("promised_ts").cast("timestamp").alias("promised_ts"),
    col("completed_ts").cast("timestamp").alias("completed_ts"),
    col("priority").cast("string").alias("priority"),
    col("sku_count").cast("int").alias("sku_count"),
    col("item_count").cast("int").alias("item_count"),
    col("status").cast("string").alias("status"),
    col("cancel_reason").cast("string").alias("cancel_reason"),
    col("customer_region").cast("string").alias("customer_region"),
    col("ingestion_timestamp").cast("timestamp").alias("ingestion_timestamp"),
    col("source_file").cast("string").alias("source_file"),
    col("load_date").cast("date").alias("load_date")
)

In [0]:
display(orders_df)

In [0]:
%sql
SELECT count(*) AS order_count,order_id
FROM bronze.orders
WHERE NOT (
    created_ts <= completed_ts
)
GROUP BY order_id;

In [0]:
%sql
SELECT *
FROM bronze.orders
WHERE order_id IN (
    SELECT order_id
    FROM bronze.orders
    GROUP BY order_id
    HAVING COUNT(*) > 1
);

In [0]:
%sql
SELECT * 
FROM bronze.orders
WHERE order_id IS NULL OR warehouse_id IS NULL;

In [0]:
%sql
SELECT 
    status,
    COUNT(*) AS order_count
FROM bronze.orders
GROUP BY status
ORDER BY order_count DESC;

In [0]:
valid_orders = orders_df.filter(col("status") != "unknown_status")

In [0]:
invalid_orders = orders_df.filter(col("status") == "unknown_status")

In [0]:
%sql
SELECT 
    warehouse_id,
    COUNT(*) AS order_count
FROM bronze.orders
GROUP BY warehouse_id
ORDER BY warehouse_id DESC;

In [0]:
%sql
SELECT 
    status,
    COUNT(*) AS order_count
FROM bronze.orders
GROUP BY status
ORDER BY order_count DESC;

In [0]:
%sql
SELECT * 
FROM bronze.orders
WHERE sku_count<1 OR item_count<1;

In [0]:
%sql
SELECT 
    priority,
    COUNT(*) AS order_count
FROM bronze.orders
GROUP BY priority
ORDER BY order_count DESC;

In [0]:
%sql
SELECT 
    customer_region,
    COUNT(*) AS order_count
FROM bronze.orders
GROUP BY customer_region
ORDER BY order_count DESC;

In [0]:
invalid_orders.write \
    .format("delta") \
    .mode("append") \
    .save("abfss://quarantine@stautofulfildata.dfs.core.windows.net/orders/orders_invalid")

In [0]:
valid_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://silver@stautofulfildata.dfs.core.windows.net/orders/orders_valid")